# Traitors chart: 4:5 version

Same design and the same `tracker.csv` as `traitors_chart.ipynb`, on a 1080x1350 canvas. Saves to `images/traitors_4x5_ep{N}.png`, so it never overwrites the square chart. Change `current_episode` in the settings cell, then run all cells.

In [ ]:
# import packages
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import RendererAgg
from matplotlib.markers import MarkerStyle
from matplotlib.font_manager import FontProperties
from PIL import Image

In [ ]:
# settings
# the episode this chart is for: columns after it show as future
current_episode = 3
n_episodes = 9

tracker_path = 'tracker.csv'
icon_path = 'images/traitorsIcon.png'
output_path = f'images/traitors_4x5_ep{current_episode}.png'

# annotations: (text, player name, episode the arrow points at, box top-left x px, box top-left y px)
# e.g. ('Rob has spotted a traitor\nevery single week', 'Rob Beckett', 5, 720, 470)
annotations = []

In [ ]:
# setup data
# tracker.csv has one row per player; update it after each episode:
#   Traitor_From_Episode  blank for faithfuls, else the episode they became a traitor (1 for original traitors)
#   Eliminated_Episode    blank while they're still in, else the episode they were banished or murdered
#   Ep1, Ep2, ...         Y if they voted for a traitor at that round table, N if not
ep_cols = [f'Ep{ep}' for ep in range(1, n_episodes + 1)]
players = pd.read_csv(tracker_path, dtype={c: 'string' for c in ep_cols})
players['Name'] = players['Name'].str.strip()

# episodes with no column in the tracker yet are treated as blank
for c in ep_cols:
    if c not in players.columns:
        players[c] = pd.Series(pd.NA, index=players.index, dtype='string')

answers = {'y': True, 'yes': True, 'n': False, 'no': False}
votes = players[ep_cols].apply(lambda col: col.str.strip().str.lower().map(answers))

# eliminations and recruitments after the current episode haven't aired, so they're ignored
players['Out'] = players['Eliminated_Episode'] <= current_episode
players['Traitor'] = players['Traitor_From_Episode'] <= current_episode

# check the tracker: anything unreadable, and any vote missing for an episode the player was in
problems = []
for i, player in players.iterrows():
    last_vote = player['Eliminated_Episode'] - 1 if player['Out'] else current_episode
    for ep in range(1, n_episodes + 1):
        raw = player[f'Ep{ep}']
        if pd.notna(raw) and pd.isna(votes.at[i, f'Ep{ep}']):
            problems.append(f"{player['Name']} Ep{ep}: '{raw}' isn't Y or N")
        elif ep <= last_vote and pd.isna(raw):
            problems.append(f"{player['Name']} Ep{ep}: no vote entered (drawn as not voting for a traitor)")
if problems:
    warnings.warn('Check tracker.csv:\n' + '\n'.join(problems))

players[ep_cols] = votes.eq(True)

def cell_status(row, episode):
    if row['Out'] and episode > row['Eliminated_Episode']:
        return None
    if row['Out'] and episode == row['Eliminated_Episode']:
        return 'Eliminated'
    if episode > current_episode:
        return 'Future'
    return 'Traitor vote' if row[f'Ep{episode}'] else 'Faithful vote'

# display names
players['Label'] = (players['Name']
                    .str.replace('Professor ', 'Prof. ', regex=False)
                    .str.replace("'", '’', regex=False))

In [ ]:
# order: players still in by surname, then eliminated players, most recent first
players['Surname'] = players['Name'].str.split().str[-1]
still_in = players[~players['Out']].sort_values('Surname')
out = players[players['Out']].sort_values(['Eliminated_Episode', 'Traitor', 'Surname'],
                                          ascending=[False, False, True])
players = pd.concat([still_in, out]).reset_index(drop=True)

In [ ]:
# style
# every measurement is in pixels on the 1080x1350 (4:5) canvas
dpi = 300
width_px, height_px = 1080, 1350
margin_px = 50
# pull the grid of dots in this far from the right and bottom margins
right_inset_px = 2
bottom_inset_px = 59.5  # leaves room above the note

def pt(px):
    """Convert canvas pixels to matplotlib points."""
    return px * 72 / dpi

background = '#F7F1E1'
ink = '#1E1F1E'
orange = '#FF8F01'
sand = '#E2C58B'
sand_light = '#EBD6AE'
maroon = '#7D0038'
annotation_stroke = '#A44D74'
grey = '#8F8F8F'
out_alpha = 0.4  # eliminated players are drawn faded

font_size = pt(22)
# numeric weights: matplotlib's 'light' means 200, which would pick Poppins Thin
light = FontProperties(family='Poppins', weight=300, size=font_size)
regular = FontProperties(family='Poppins', weight=400, size=font_size)
semibold = FontProperties(family='Poppins', weight=600, size=font_size)
cap_offset = 0.35 * 22  # baseline sits this far below the header row's centre, centring cap height
row_text_offset = 10.2  # names sit a little lower than the dots they label
legend_text_offset = 11.5  # likewise the legend labels, against their icons

# note along the bottom margin, below the grid
note = 'Note: Traitors logo/branding are properties of IDTV, Studio Lambert and the BBC.'
note_font = FontProperties(family='Poppins', weight=300, size=pt(13.6))
note_bottom = height_px - 45

measure = RendererAgg(width_px, height_px, dpi)

def text_width(s, prop):
    """Width of a piece of text in canvas pixels."""
    return measure.get_text_width_height_descent(s, prop, ismath=False)[0]

# marks
icon_px = 36
dot = dict(traitor=35, small=12)
vote_outline = 1.25
x_marker = MarkerStyle('x', capstyle='round')
x_size, x_width = 24, 6.5  # the X in the grid
legend_mark = dict(traitor=25, x=18)  # the legend's circle and X are smaller than the grid's

# layout
name_x = margin_px + icon_px + 9
legend_y = 76.5  # centre of the legend row
legend_gap = 43  # space between one legend item and the next
eliminated_legend_x = 493  # left edge of the legend's X, unless the label before it runs longer
header_y = 157

# episode columns run from col_first_x out to the grid's right edge, however many episodes there are;
# a vote dot in the last column just touches that edge
grid_right = width_px - margin_px - right_inset_px
grid_bottom = height_px - margin_px - bottom_inset_px
col_first_x = 422.5
col_last_x = grid_right - dot['traitor'] / 2
col_pitch = (col_last_x - col_first_x) / max(n_episodes - 1, 1)

# player rows run from row_y0 down to the grid's bottom edge; the tallest thing in the bottom row
# is a vote dot, or the traitor icon if that player is a traitor
row_y0 = 202
bottom_half = max(dot['traitor'], icon_px if players['Traitor'].iloc[-1] else 0) / 2
row_pitch = (grid_bottom - bottom_half - row_y0) / max(len(players) - 1, 1)

def col_x(episode):
    return col_first_x + (episode - 1) * col_pitch

def row_y(i):
    return row_y0 + i * row_pitch

# icon, cropped to the logo and resized to the exact pixel size
icon = Image.open(icon_path).convert('RGBA')
icon = icon.crop(icon.getchannel('A').getbbox())
icon_w = round(icon_px * icon.width / icon.height)
icon = np.asarray(icon.resize((icon_w, icon_px), Image.LANCZOS))

In [ ]:
# create plot: one axes over the whole canvas, in pixel coordinates with y pointing down
fig = plt.figure(figsize=(width_px / dpi, height_px / dpi), dpi=dpi, facecolor=background)
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, width_px)
ax.set_ylim(height_px, 0)
ax.axis('off')

def draw_icon(x, y_centre, alpha=1):
    y0 = round(y_centre - icon_px / 2)
    ax.imshow(icon, extent=[x, x + icon_w, y0 + icon_px, y0], alpha=alpha,
              interpolation='nearest', aspect='auto', zorder=3)

def draw_dot(x, y, kind, alpha=1, size=None):
    """size overrides the default diameter of the circle, or the extent of the X."""
    if kind == 'Traitor vote':
        ax.scatter(x, y, s=pt(size or dot['traitor']) ** 2, c=orange, edgecolors=ink, linewidths=pt(vote_outline),
                   alpha=alpha, zorder=3)
    elif kind == 'Faithful vote':
        ax.scatter(x, y, s=pt(dot['small']) ** 2, c=sand, linewidths=0, alpha=alpha, zorder=3)
    elif kind == 'Future':
        ax.scatter(x, y, s=pt(dot['small']) ** 2, c=background, edgecolors=sand_light, linewidths=pt(1), zorder=3)
    elif kind == 'Eliminated':
        ax.plot(x, y, marker=x_marker, markersize=pt(size or x_size), markeredgewidth=pt(x_width),
                color=maroon, alpha=alpha, linestyle='none', zorder=3)

# legend
x = margin_px
legend = [('icon', 'Traitor'), ('Traitor vote', 'Voted for a traitor'), ('Eliminated', 'Eliminated')]
for marker, label in legend:
    if marker == 'icon':
        draw_icon(x, legend_y)
        x += icon_w
    else:
        mark_size = legend_mark['traitor'] if marker == 'Traitor vote' else legend_mark['x']
        outer = mark_size + (vote_outline if marker == 'Traitor vote' else x_width)
        if marker == 'Eliminated':
            x = max(x - legend_gap / 2, eliminated_legend_x)
        draw_dot(x + outer / 2, legend_y, marker, size=mark_size)
        x += outer
    x += 9
    ax.text(x, legend_y + legend_text_offset, label, fontproperties=light, color=ink, va='baseline')
    x += text_width(label, light) + legend_gap

# episode headers, centred on their column; a header wider than the space left before
# the right margin (the finale's bold 'Ep N') is nudged left to stay inside it
for ep in range(1, n_episodes + 1):
    if ep == current_episode:
        label, prop, colour = f'Ep {ep}', semibold, ink
    else:
        label, prop, colour = str(ep), light, ink if ep < current_episode else sand
    x = min(col_x(ep), width_px - margin_px - text_width(label, prop) / 2)
    ax.text(x, header_y + cap_offset, label, fontproperties=prop, color=colour, ha='center', va='baseline')

# players: traitors get the icon and a bold name, eliminated players are faded
for i, player in players.iterrows():
    y = row_y(i)
    alpha = out_alpha if player['Out'] else 1
    if player['Traitor']:
        draw_icon(margin_px, y, alpha)
    prop = semibold if player['Traitor'] else light
    ax.text(name_x, y + row_text_offset, player['Label'], fontproperties=prop, color=ink, alpha=alpha, va='baseline')

    for ep in range(1, n_episodes + 1):
        status = cell_status(player, ep)
        if status is None:
            continue
        # this week's elimination stays at full strength
        cell_alpha = 1 if status == 'Eliminated' and ep == current_episode else alpha
        draw_dot(col_x(ep), y, status, cell_alpha)

# note, lined up with the traitor icons, its bottom on the bottom margin
ax.text(margin_px, note_bottom, note, fontproperties=note_font, color=grey, va='bottom')

# annotations
# matplotlib steps lines by the font's own line height x linespacing, so measure it to get 30px
renderer = fig.canvas.get_renderer()
one_line, two_lines = (ax.text(0, 0, s, fontproperties=regular, linespacing=1) for s in ('A', 'A\nA'))
font_line_px = (two_lines.get_window_extent(renderer).height
                - one_line.get_window_extent(renderer).height) * width_px / fig.bbox.width
one_line.remove(), two_lines.remove()
line_spacing = 30 / font_line_px

row_of = {name: i for i, name in enumerate(players['Name'])}
for text, name, ep, box_x, box_y in annotations:
    pad = 12
    ax.annotate(text, xy=(col_x(ep), row_y(row_of[name])), xytext=(box_x + pad, box_y + pad),
                fontproperties=regular, color=maroon, linespacing=line_spacing, ha='left', va='top',
                bbox=dict(boxstyle=f'square,pad={pad / 22}', facecolor='#FFFFFF',
                          edgecolor=annotation_stroke, linewidth=pt(1.5)),
                arrowprops=dict(arrowstyle='-', color=annotation_stroke, linewidth=pt(1.5),
                                shrinkA=0, shrinkB=pt(dot['traitor'] / 2 + 4)),
                zorder=4)

# save final plot
fig.savefig(output_path, dpi=dpi, facecolor=background)
plt.show()